# kNN implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq imbalanced-learn matplotlib numpy pandas scikit-learn seaborn


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from imblearn.over_sampling import RandomOverSampler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler


### 03. Data Loading and Preprocessing


Looking at the dataset that will be processed, this is the **MAGIC Gamma Telescope Dataset** which contains data from a ground-based atmospheric Cherenkov gamma telescope. The dataset is used for classification of high energy gamma particles from hadrons (background noise).

**Dataset characteristics:**
- **Total samples**: 19,020 observations
- **Features**: 10 continuous variables measuring telescope imaging parameters
- **Target**: Binary classification (Gamma particles vs Hadrons)
    - Class 1: Gamma particles (signal)
    - Class 0: Hadrons (background)

**Feature descriptions:**
- `fLength`: Major axis of ellipse [mm]
- `fWidth`: Minor axis of ellipse [mm]  
- `fSize`: 10-log of sum of content of all pixels [in #phot]
- `fConc`: Ratio of sum of two highest pixels over fSize [ratio]
- `fConc1`: Ratio of highest pixel over fSize [ratio]
- `fAsym`: Distance from highest pixel to center, projected onto major axis [mm]
- `fM3Long`: 3rd root of third moment along major axis [mm]
- `fM3Trans`: 3rd root of third moment along minor axis [mm]
- `fAlpha`: Angle of major axis with vector to origin [deg]
- `fDist`: Distance from origin to center of ellipse [mm]

The goal is to distinguish between gamma-ray showers (which are of astrophysical interest) and hadronic showers initiated by cosmic rays in the upper atmosphere (which represent background noise).


In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/magic_gamma_telescope/magic04.data', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1', 'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Mapping the classes to numerical values

dataframe['class'] = dataframe['class'].map({'g': 1, 'h': 0})
dataframe.head()


### 04. Data Visualization


In [ ]:
for label in columns_name[:-1]:
    plt.figure(figsize = (10, 6))

    plt.hist(dataframe[dataframe['class'] == 0][label], label = 'Hadron (h)', density = True, alpha = 0.5, color = 'blue')
    plt.hist(dataframe[dataframe['class'] == 1][label], label = 'Gamma (g)', density = True, alpha = 0.5, color = 'orange')

    plt.title(f'Distribution of {label} by Class')
    plt.xlabel(label)
    plt.ylabel('Probability')
    plt.legend(title = 'Class')
    plt.grid()

    plt.show()


### 05. Dataset Splitting and Scaling


In [ ]:
# Shuffling the dataframe

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining a function to scale the data and oversample if necessary

def scale_data(dataframe, oversample = False):
    features = dataframe.columns[:-1]
    target = dataframe.columns[-1]

    X = dataframe[features].values
    y = dataframe[target].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if oversample:
        ros = RandomOverSampler(random_state = 42)
        X_resampled, y_resampled = ros.fit_resample(X_scaled, y)
        dataframe_scaled = np.hstack((X_resampled, np.reshape(y_resampled, (-1, 1))))
        return dataframe_scaled, X_resampled, y_resampled
    else:
        dataframe_scaled = np.hstack((X_scaled, np.reshape(y, (-1, 1))))
        return dataframe_scaled, X_scaled, y


In [ ]:
# Defining the train, validation and test datasets sizes

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes before scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


In [ ]:
# Scaling and oversampling the datasets

train_dataset, X_train, y_train = scale_data(train_dataset, oversample = True)
validation_dataset, X_validation, y_validation = scale_data(validation_dataset, oversample = False)
test_dataset, X_test, y_test = scale_data(test_dataset, oversample = False)

print('\nDatasets sizes after scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


### 06. kNN Implementation and Evaluation


In [ ]:
# kNN implementation

for number_neighbors in range(1, 5, 2):
    knn_model = KNeighborsClassifier(n_neighbors = number_neighbors)
    knn_model.fit(X_train, y_train)

    y_predictions_validation = knn_model.predict(X_validation)
    y_predictions_test = knn_model.predict(X_test)

    print(f'\nNumber of Neighbors: {number_neighbors}')
    print('Validation Set Classification Report:')
    print(classification_report(y_validation, y_predictions_validation))

    print('Test Set Classification Report:')
    print(classification_report(y_test, y_predictions_test))
